# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided exploration of the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # access metadata as an object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (`@id`), their fields (`@id`), and columns in the dataset. All entities are referenced by their `@id` fields.

In [ ]:
# Show available record sets

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets discovered in metadata. Inspecting Distribution objects instead...")
    print("Distributions in metadata:")
    for dist in getattr(metadata, 'distribution', []):
        print(f"  Distribution @id: {dist.get('@id', None)}")
    print("\nTry to access the default record set inferred by mlcroissant:")
    default_record_set = dataset.default_record_set
    print(f"Default inferred record set: {default_record_set}")

    # Print example records
    print("\nFirst 2 records in the default record set:")
    for i, rec in enumerate(dataset.records(record_set=default_record_set)):
        print(rec)
        if i >= 1:
            break
    print("\nAvailable columns/keys in the records:")
    print(list(rec.keys()))

else:
    print("Record Sets found in metadata:")
    for rs in record_sets:
        print(f"  @id: {rs.id}")
        print(f"  name: {rs.name}")
        print("  Field @ids and types:")
        for field in rs.fields:
            print(f"   - {field.id} ({field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data assuming default_record_set inferred
record_sets_to_load = []
if hasattr(dataset, 'default_record_set') and dataset.default_record_set is not None:
    # Use the inferred record set
    record_sets_to_load = [dataset.default_record_set]
else:
    # Use all discovered record sets
    record_sets_to_load = [rs.id for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    else:
        print(f"No records found for record set {record_set_id}")

# Inspect columns of the first DataFrame
example_record_set_id = record_sets_to_load[0]
print(f"\nColumns in record set '{example_record_set_id}':")
print(dataframes[example_record_set_id].columns.tolist())

dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, or grouping data by key attributes using `@id` field references.

In [ ]:
# You may need to select the exact field (@id/column) names from the previous overview section.
# For this dataset, example numeric columns include 'MW [kDa]' (molecular weight),
# 'Coverage (%)', or peptide-related counts.

df = dataframes[example_record_set_id]

# Pick a numeric field based on the available columns.
numeric_field = None
for col in df.columns:
    # Try to identify a sensible numeric field. Adjust if needed.
    if 'MW' in col or 'weight' in col or 'Coverage' in col:
        numeric_field = col
        break
if numeric_field is None and len(df.select_dtypes(include='number').columns) > 0:
    numeric_field = df.select_dtypes(include='number').columns[0]
if numeric_field is None:
    print("No numeric field found for EDA.")

if numeric_field is not None:
    print(f"Selected numeric field: {numeric_field}")

    # Convert the field to numeric (may require cleaning if strings)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()  # set threshold as mean for demo purposes
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records (first rows):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical column, e.g., 'description', 'Accession' or similar
    possible_group_fields = [col for col in df.columns if col.lower() in ['description', 'accession', 'protein'] or 'group' in col.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None

    if group_field:
        print(f"\nGrouping by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
        print(grouped_df.head())
else:
    print("Exploratory Data Analysis skipped due to lack of numeric fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using pandas/Matplotlib. This example shows the distribution of a numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using the `mlcroissant` library:

- Loaded dataset metadata and reviewed record structure using Croissant schema and `@id` fields
- Extracted core protein records and viewed their numerical distribution (e.g., molecular weights)
- Demonstrated basic filtering, normalization, and grouping by categorical fields
- Visualized the distribution of a key numeric variable

**Key takeaways:** The dataset provides structured protein abundance and identification information suitable for further quantitative or comparative proteomics analysis. The explicit use of `@id` field references ensures robust, unambiguous data processing.